# QPGP base functions

In [ ]:

import numpy as np
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from stretch_body.robot import Robot
from numpy.random import multivariate_normal as mvnrnd
from scipy.linalg import toeplitz, solve_triangular, solve_toeplitz, cholesky, inv
from numpy.fft import fft, ifft

def QPGP_sim(n, p, w,Kappa):
    """
    Simulates a quasi-periodic Gaussian process (QPGP).

    Parameters:
    - n     : int, total length of time series
    - p     : int, periodicity parameter
    - w : float, autoregressive weight
    - Kappa  : covariance matrix

    Returns:
    - X     : np.ndarray of shape (n,), simulated QPGP series
    """

    # Number of blocks
    k = n // p

    # Preallocate X
    if n % p == 0:
        X = np.zeros(n)
        # First block ~ N(0, Kappa / (1 - w^2))
        X[:p] = mvnrnd(mean=np.zeros(p), cov=Kappa / (1 - w**2))
        for i in range(1, k):
            prev = X[(i-1)*p : i*p]
            noise = mvnrnd(mean=np.zeros(p), cov=Kappa)
            X[i*p : (i+1)*p] = w * prev + noise
    else:
        X = np.zeros((k+1)*p)
        X[:p] = mvnrnd(mean=np.zeros(p), cov=Kappa / (1 - w**2))
        for i in range(1, k+1):
            prev = X[(i-1)*p : i*p]
            noise = mvnrnd(mean=np.zeros(p), cov=Kappa)
            X[i*p : (i+1)*p] = w * prev + noise
        X = X[:n]

    return X

def logL(n, p, w, Kappa, X):
    """
    Computes the log-likelihood for block-wise Gaussian model using
    Toeplitz Cholesky factorization.

    Parameters:
    - n     : int, total length of vector X
    - p     : int, block size
    - w     : float, autoregressive coefficient
    - Kappa : np.ndarray (p x p), Toeplitz covariance matrix
    - X     : np.ndarray of shape (n,), data vector

    Returns:
    - logL  : float, computed log-likelihood
    """
    Kappa = Kappa + 1e-10 * np.eye(p)
    c = Kappa[:, 0]
    L = cholesky(Kappa, lower=True)

    # Number of blocks
    k = n // p
    sum_term = 0.0

    # Loop over blocks
    for i in range(1, k):
        X_curr = X[i*p : (i+1)*p]
        X_prev = X[(i-1)*p : i*p]
        diff = X_curr - w * X_prev

        # Efficiently solve Kappa^{-1} diff
        sol = solve_toeplitz((c, c), diff)
        sum_term += diff @ sol    # Compute log-determinant from Cholesky diagonal

    logdet_Kappa = 2 * np.sum(np.log(np.diag(L)))

    # Log-likelihood (negative)
    log_likelihood = ((k - 1) * p * np.log(2 * np.pi)) / 2 + ((k - 1) * logdet_Kappa) / 2 + sum_term

    # Handle numerical issues
    if np.isnan(log_likelihood) or log_likelihood < -1e10:
        return np.inf
    else:
        return log_likelihood


def w_est(n, p, Kappa, X):
    """
    Estimates the autoregressive parameter omega.

    Parameters:
    - n     : int, number of observations
    - p     : int, block size
    - Kappa : np.ndarra
    y (p x p), covariance matrix
    - X     : np.ndarray (n,), data vector

    Returns:
    - w_est : float, estimated omega
    """
    k = n // p

    # Spectral decomposition of Kappa
    eig_vals, U = np.linalg.eigh(Kappa)
    Kappa_inv = U @ np.diag(1.0 / eig_vals) @ U.T

    # Numerator (i1i)
    i1i = 0.0
    for i in range(1, k):
        x1 = X[(i-1)*p - p : (i-1)*p] if i > 1 else X[0:p]
        x2 = X[(i)*p - p : (i)*p]
        i1i += x1.T @ Kappa_inv @ x2

    # Denominator (ii)
    ii = 0.0
    for i in range(k - 1):
        x = X[i*p : (i+1)*p]
        ii += x.T @ Kappa_inv @ x

    # If n is not divisible by p, include the remainder part
    if n % p != 0:
        Kappastar = Kappa[0:(n - k*p), 0:(n - k*p)]
        eig_vals_star, U_star = np.linalg.eigh(Kappastar)
        Kappastar_inv = U_star @ np.diag(1.0 / eig_vals_star) @ U_star.T

        xi = X[k*p - p : n - p]
        xi1 = X[k*p : n]
        nstar = xi.T @ Kappastar_inv @ xi1
        dstar = xi.T @ Kappastar_inv @ xi
    else:
        nstar = 0.0
        dstar = 0.0

    # Final omega estimate
    return (i1i + nstar) / (ii + dstar)


def Kappa_est(n, p, omega, X):
    """
    Direct translation of MATLAB Kappa_est.
    Inputs:
        n     : Number of observations
        p     : Block size
        omega : Weight parameter
        X     : Data vector (1D numpy array)
    Output:
        Kappa_est : Estimated covariance matrix (p x p)
    """
    k = n // p             # floor(n / p)
    l = n - k * p          # remainder length

    S = np.zeros((p, p))   # Initialize S matrix
    S_new = np.zeros((p, p))

    # Create Y matrix from X (reshape like MATLAB)
    Y = X[:k*p].reshape((p, k), order='F')
    if l > 0:
        last_col = np.full((p, 1), np.nan)
        last_col[:l, 0] = X[k*p:]
        Y = np.hstack((Y, last_col))

    # Difference matrix
    Y_matrix = Y[:, 1:] - omega * Y[:, :-1]

    # Loop through blocks to compute S
    for i in range(1, k):
        x_current = X[i*p : (i+1)*p]
        x_prev = X[(i-1)*p : i*p]
        residual = x_current - omega * x_prev
        S += np.outer(residual, residual)

    if n % p == 0:
        S /= (k - 1)
    else:
        S = np.zeros((p, p))

        Y_matrix1 = Y_matrix[:l, :k-1]
        Y_matrix2 = Y_matrix[l:, :k-1]

        S11 = (Y_matrix1 @ Y_matrix1.T) / (k - 1)
        S12 = (Y_matrix1 @ Y_matrix2.T) / (k - 1)
        S21 = S12.T
        S22 = (Y_matrix2 @ Y_matrix2.T) / (k - 1)

        add = np.outer(Y[:l, k], Y[:l, k])
        S11star = ((k - 1) * S11 + add) / k

        S_new[:l, :l] = S11star
        S_new[:l, l:] = S11star @ inv(S11) @ S12
        S_new[l:, :l] = S_new[:l, l:].T
        S_new[l:, l:] = S22 - S21 @ inv(S11) @ S12 \
                        + S21 @ inv(S11) @ S11star @ inv(S11) @ S12

        S = S_new

    # Enforce Toeplitz structure by averaging diagonals
    m, n_ = S.shape
    for d in range(-(m-1), n_):
        indices = [(i, i - d) for i in range(m) if 0 <= i - d < n_]
        diag_vals = [S[i, j] for i, j in indices]
        avg_val = np.mean(diag_vals)
        for i, j in indices:
            S[i, j] = avg_val

    # Ensure positive definiteness via spectral fix
    epsilon = 1e-6
    f = np.real(fft(S[0, :]))
    f[f < 0] = 0
    f = np.abs(f)
    f = np.maximum(f, epsilon)
    v = np.real(ifft(f))

    return toeplitz(v)

def w_Kappa_est(n, p, X, max_iter=200, tol=1e-5):
    """
    Iteratively estimates omega and Kappa covariance matrix.

    Parameters:
    - n        : int, number of observations
    - p        : int, block size
    - X        : np.ndarray (n,), data vector
    - max_iter : int, max number of iterations (default 200)
    - tol      : float, convergence tolerance (default 1e-5)

    Returns:
    - result : dict with keys:
        'w'     : estimated omega
        'Kappa' : estimated covariance matrix
    """
    Kappa_hat = []
    w_hat = []
    l =[]

    Kappa_hat.append(np.eye(p))
    w_hat.append(w_est(n, p, Kappa_hat[0], X))

    for i in range(1, max_iter):
       # print(f"Loop iteration: {i}")
        a = Kappa_est(n, p, w_hat[i-1], X)
        Kappa_hat.append(a)
        b = w_est(n, p, Kappa_hat[i], X)
        w_hat.append(b)
        c=logL(n, p, b, a, X)
        l.append(c)
        if i > 1 and abs(l[-1] - l[-2]) < tol:
            break

    last_iter = i
    w_final = w_hat[last_iter]
    Kappa_final = Kappa_hat[last_iter]

    return {'w': w_final, 'Kappa': Kappa_final}


def par_est(n, p_search, X):
    """
    Estimate the best (p, omega, Kappa) by maximizing the log-likelihood.

    Parameters:
    - n         : int, length of data
    - p_search  : list or array of candidate p values
    - X         : (n,) numpy array of data

    Returns:
    - result: dict with keys:
        - 'p'      : estimated p
        - 'w'      : estimated omega
        - 'Kappa'  : estimated Kappa matrix (2D numpy array)
    """
    logL_vals = []

    for p in p_search:
        var = w_Kappa_est(n, p, X)
        w = var['w']
        Kappa = var['Kappa']
        logL_val = logL(n, p, w, Kappa, X)
        logL_vals.append(logL_val)

    best_idx = np.argmin(logL_vals)
    p_est = p_search[best_idx]

    final = w_Kappa_est(n, p_est, X)
    result = {
        'p': p_est,
        'w': final['w'],
        'Kappa': final['Kappa']
    }

    return result


def pred_element(n, p, w, Kappa, X):
    pred = np.zeros(n + p)
    Kappa = Kappa + 1e-10 * np.eye(p)  # Numerical stability

    for j in range(1, n + p + 1):
        if j % p != 0:
            i = j // p
            l = j - i * p
        else:
            i = j // p - 1
            l = p

        # Skip if indices invalid or l==1 (no conditioning)
        if l == 1 or i < 0:
            pred1 = w * X[(i - 1) * p + l - 1] if j > p and i - 1 >= 0 else 0
            pred2 = 0
            pred[j - 1] = pred1 + pred2
            continue

        # Valid case
        if j > p:
            pred1 = w * X[(i - 1) * p + l - 1]
            K11 = Kappa[l - 1, :l - 1].reshape(1, -1)
            K00 = Kappa[:l - 1, :l - 1]

            X_curr = X[i * p : i * p + l - 1]
            X_prev = X[(i - 1) * p : (i - 1) * p + l - 1]

            if X_curr.size == 0 or X_prev.size == 0 or K00.shape[0] == 0:
                # Skip problematic step
                pred2 = 0
            else:
                delta_X = (X_curr - w * X_prev).reshape(-1, 1)
                pred2 = (K11 @ np.linalg.inv(K00) @ delta_X).item()
        else:
            pred1 = 0
            K11 = Kappa[l - 1, :l - 1].reshape(1, -1)
            K00 = Kappa[:l - 1, :l - 1]
            X_prev = X[:l - 1]

            if X_prev.size == 0 or K00.shape[0] == 0:
                pred2 = 0
            else:
                pred2 = (K11 @ np.linalg.inv(K00) @ X_prev.reshape(-1, 1)).item()

        pred[j - 1] = pred1 + pred2

    return pred


def pred_block(n, p, w, X):
   # pred = np.zeros(p)
    pred = w*X[-p:]

    return pred



# Desired trajectory

In [ ]:
lift_min, lift_max = 0.05, 0.25
arm_min, arm_max = 0.25, 0.65

N = 100
a, b = 3, 2
delta = np.pi / 2
scale_factor = 0.5

arm_center = (arm_min + arm_max)/2
lift_center = (lift_min + lift_max)/2
arm_amp = (arm_max - arm_min)/2 * 0.8 * scale_factor
lift_amp = (lift_max - lift_min)/2 * 0.8 * scale_factor

t = np.linspace(0, 2*np.pi, N, endpoint=False)
base_trajectory = np.zeros((N,2))
base_trajectory[:,0] = arm_center + 0.5*arm_amp*np.sin(a*t + delta)
base_trajectory[:,1] = lift_center + 0.5*lift_amp*np.sin(b*t)
base_trajectory[:,0] -= 0.2
base_trajectory[:,1] += 0.3


# Standard ILC

In [ ]:
# ==============================
# ILC parameters
# ==============================
iterations = 50
sleep_time = 0.1
L_gain = 0.05
Kp = 1.5 # predictive gain for QPGP

# ==============================
# Initialize robot
# ==============================
robot = Robot()
robot.startup()
robot.stow()
time.sleep(5)
robot.head.pose('ahead')
robot.end_of_arm.move_to('wrist_pitch', 1.57)
robot.end_of_arm.move_to('wrist_yaw', 1.57)
robot.push_command()
time.sleep(1)

# ==============================
# Histories for trajectories and errors
# ==============================
traj_std_hist, traj_qp_hist = [], []
err_std_hist, err_qp_hist = [], []

# ==============================
# --- Standard ILC loop ---
# ==============================
print("=== Standard ILC ===")
u_traj_std = base_trajectory.copy()

for k in range(iterations):
    print(f"\n--- Iteration {k+1} ---")
    actual_std = []

    # Move to first point
    robot.arm.move_to(u_traj_std[0,0])
    robot.lift.move_to(u_traj_std[0,1])
    robot.push_command()
    time.sleep(2)

    for i in range(N):
        target_y, target_z = u_traj_std[i]
        robot.arm.move_to(target_y)
        robot.lift.move_to(target_z)
        robot.push_command()
        time.sleep(sleep_time)

        act_y = robot.arm.status['pos']
        act_z = robot.lift.status['pos']
        actual_std.append([act_y, act_z])

        # Update trajectory
        err_vec = base_trajectory[i] - np.array([act_y, act_z])
        u_traj_std[i] += L_gain * err_vec

        print(f"[Standard] Iter {k+1}, Point {i+1}: "
              f"Ref=({base_trajectory[i,0]:.3f},{base_trajectory[i,1]:.3f}) | "
              f"Actual=({act_y:.3f},{act_z:.3f}) | "
              f"Error={np.linalg.norm(err_vec):.4f}")

    # Save iteration history
    traj_std_hist.append(np.array(actual_std))
    err_std_hist.append(np.linalg.norm(base_trajectory - np.array(actual_std), axis=1))


# QPGP-PILC

In [ ]:
# ==============================
# --- QPGP ILC loop ---
# ==============================
u_arm_qp = base_trajectory[:,0].copy()
u_lift_qp = base_trajectory[:,1].copy()
d_arm_hist, d_lift_hist = [], []

for k in range(iterations):
    print(f"\n--- Iteration {k+1} ---")
    actual_qp = []
    d_arm = np.zeros(N)
    d_lift = np.zeros(N)

    # Move to first point
    robot.arm.move_to(u_arm_qp[0])
    robot.lift.move_to(u_lift_qp[0])
    robot.push_command()
    time.sleep(2)

    for i in range(N):
        robot.arm.move_to(u_arm_qp[i])
        robot.lift.move_to(u_lift_qp[i])
        robot.push_command()
        time.sleep(sleep_time)

        act_y = robot.arm.status['pos']
        act_z = robot.lift.status['pos']
        actual_qp.append([act_y, act_z])

        # Baseline ILC update
        err_vec = np.array([base_trajectory[i,0]-act_y,
                            base_trajectory[i,1]-act_z])
        u_arm_qp[i] += L_gain * err_vec[0]
        u_lift_qp[i] += L_gain * err_vec[1]
        d_arm[i] = L_gain * err_vec[0]
        d_lift[i] = L_gain * err_vec[1]

        print(f"[QPGP] Iter {k+1}, Point {i+1}: "
              f"Ref=({base_trajectory[i,0]:.3f},{base_trajectory[i,1]:.3f}) | "
              f"Actual=({act_y:.3f},{act_z:.3f}) | "
              f"Error={np.linalg.norm(err_vec):.4f}")

    # Predictive QPGP correction
    if k >= 1 and Kp > 0 and len(d_arm_hist) > 1:
        d1_hist = np.concatenate(d_arm_hist)[-2*N:]
        d2_hist = np.concatenate(d_lift_hist)[-2*N:]

        par1 = par_est(len(d1_hist), [N], d1_hist)
        par2 = par_est(len(d2_hist), [N], d2_hist)

        u_arm_qp += Kp * pred_block(len(d1_hist), N, par1['w'], d1_hist)
        u_lift_qp += Kp * pred_block(len(d2_hist), N, par2['w'], d2_hist)

    d_arm_hist.append(d_arm)
    d_lift_hist.append(d_lift)

    # Save iteration history
    traj_qp_hist.append(np.array(actual_qp))
    err_qp_hist.append(np.linalg.norm(base_trajectory - np.array(actual_qp), axis=1))

# ==============================
# Final stow
# ==============================
robot.stow()
time.sleep(1)
robot.stop()

# Plots and figures

In [ ]:
# ==============================
# PLOTS
# ==============================
# 1. Mean absolute error per iteration
mae_std = [np.mean(err) for err in err_std_hist]
mae_qp  = [np.mean(err) for err in err_qp_hist]

import matplotlib.pyplot as plt

# Set figure size for double-column IEEE (~7.16 in width, ~3.5 in height)
plt.figure(figsize=(7.16, 3.5))

# Plot MAE per iteration
plt.plot(range(1, iterations+1), mae_std, '-', label='Standard ILC')
plt.plot(range(1, iterations+1), mae_qp, '-', label='QPGP ILC')

# Labels and title
plt.xlabel('Iteration No.')
plt.ylabel('Mean Absolute Error (m)')
plt.title('Mean Absolute Error Comparison')

# Legend and grid
plt.legend()
plt.grid(True)

# Save figure with high resolution
plt.savefig('mae_per_iteration.pdf', dpi=300, bbox_inches='tight')  # PDF preferred for IEEE

plt.show()


# 2. Concatenated errors
concat_std = np.concatenate(err_std_hist)
concat_qp = np.concatenate(err_qp_hist)

plt.figure()
plt.plot(concat_std, label='Standard ILC')
plt.plot(concat_qp, label='QPGP ILC')
plt.xlabel('Point index across iterations')
plt.ylabel('Error')
plt.title('Concatenated errors across iterations')
plt.legend()
plt.grid(True)
plt.savefig('concatenated_errors.png', dpi=300)

# 3. Trajectories at 15th iteration

plt.figure()
plt.plot(base_trajectory[:,0], base_trajectory[:,1], 'k-', label='Reference')
plt.plot(traj_std_hist[4][:,0], traj_std_hist[4][:,1], 'r--', label='Standard ILC')
plt.plot(traj_qp_hist[4][:,0], traj_qp_hist[4][:,1], 'b--', label='QPGP PILC')
plt.xlabel('Arm')
plt.ylabel('Lift')
plt.title('Trajectories at 5th iteration')
plt.legend()
plt.grid(True)
plt.savefig('trajectories_iter5.png', dpi=300)

# 4. Trajectories at 10th iteration

# Set figure size for IEEE double-column (~7.16 in width, ~3.5 in height)
plt.figure(figsize=(7.16, 3.5))

# Plot trajectories at 10th iteration
plt.plot(base_trajectory[:,0], base_trajectory[:,1], 'k-', label='Reference')
plt.plot(traj_std_hist[9][:,0], traj_std_hist[9][:,1], 'r--', label='Standard ILC')
plt.plot(traj_qp_hist[9][:,0], traj_qp_hist[9][:,1], 'b--', label='QPGP PILC')

# Labels and title
plt.xlabel('Arm')
plt.ylabel('Lift')
plt.title('Trajectories at 10th iteration')

# Legend and grid
plt.legend(fontsize='small', loc='best')
plt.grid(True)

# Save figure with high resolution
plt.savefig('trajectories_iter10.pdf', dpi=300, bbox_inches='tight')  # PDF preferred for IEEE

plt.show()

print("\n✅ Plots saved: mae_per_iteration.png, concatenated_errors.png, trajectories_iter5.png, trajectories_iter10.png")

import matplotlib.pyplot as plt

# Set figure size for IEEE double-column (~7.16 in width, ~3.5 in height)
plt.figure(figsize=(7.16, 3.5))

# Plot trajectories at 10th iteration
plt.plot(base_trajectory[:,0], base_trajectory[:,1], 'k-', label='Reference')
plt.plot(traj_std_hist[14][:,0], traj_std_hist[14][:,1], 'r--', label='Standard ILC')
plt.plot(traj_qp_hist[14][:,0], traj_qp_hist[14][:,1], 'b--', label='QPGP PILC')

# Labels and title
plt.xlabel('Arm')
plt.ylabel('Lift')
plt.title('Trajectories at 15th iteration')

# Legend and grid
plt.legend(fontsize='small', loc='best')
plt.grid(True)

# Save figure with high resolution
plt.savefig('trajectories_iter15.pdf', dpi=300, bbox_inches='tight')  # PDF preferred for IEEE
plt.show()

plt.figure(figsize=(7.16, 3.5))

# Plot trajectories at 10th iteration
plt.plot(base_trajectory[:,0], base_trajectory[:,1], 'k-', label='Reference')
plt.plot(traj_std_hist[19][:,0], traj_std_hist[19][:,1], 'r--', label='Standard ILC')
plt.plot(traj_qp_hist[19][:,0], traj_qp_hist[19][:,1], 'b--', label='QPGP PILC')

# Labels and title
plt.xlabel('Arm')
plt.ylabel('Lift')
plt.title('Trajectories at 20th iteration')

# Legend and grid
plt.legend(fontsize='small', loc='best')
plt.grid(True)

# Save figure with high resolution
plt.savefig('trajectories_iter20.pdf', dpi=300, bbox_inches='tight')  # PDF preferred for IEEE
plt.show()


plt.figure(figsize=(7.16, 3.5))

# Plot trajectories at 10th iteration
plt.plot(base_trajectory[:,0], base_trajectory[:,1], 'k-', label='Reference')
plt.plot(traj_std_hist[24][:,0], traj_std_hist[24][:,1], 'r--', label='Standard ILC')
plt.plot(traj_qp_hist[24][:,0], traj_qp_hist[24][:,1], 'b--', label='QPGP PILC')

# Labels and title
plt.xlabel('Arm')
plt.ylabel('Lift')
plt.title('Trajectories at 25th iteration')

# Legend and grid
plt.legend(fontsize='small', loc='best')
plt.grid(True)

# Save figure with high resolution
plt.savefig('trajectories_iter25.pdf', dpi=300, bbox_inches='tight')  # PDF preferred for IEEE
plt.show()


import pandas as pd

# Prepare data for saving
# Standard ILC
std_data = []
for k in range(iterations):
    df_iter = pd.DataFrame(traj_std_hist[k], columns=['Arm_actual', 'Lift_actual'])
    df_iter['Arm_ref'] = base_trajectory[:,0]
    df_iter['Lift_ref'] = base_trajectory[:,1]
    df_iter['Iteration'] = k + 1
    std_data.append(df_iter)
df_std_all = pd.concat(std_data, ignore_index=True)

# QPGP ILC
qp_data = []
for k in range(iterations):
    df_iter = pd.DataFrame(traj_qp_hist[k], columns=['Arm_actual', 'Lift_actual'])
    df_iter['Arm_ref'] = base_trajectory[:,0]
    df_iter['Lift_ref'] = base_trajectory[:,1]
    df_iter['Iteration'] = k + 1
    qp_data.append(df_iter)
df_qp_all = pd.concat(qp_data, ignore_index=True)

# Save to Excel with two sheets
excel_file = 'ILC_results.xlsx'

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    df_std_all.to_excel(writer, sheet_name='Standard_ILC', index=False)
    df_qp_all.to_excel(writer, sheet_name='QPGP_ILC', index=False)

print("✅ Saved all reference and actual points to 'ILC_results.xlsx'")
